# 12. Pipeline Architecture & Performance Scaling: Beginner Guide

### 🌟 What is Pipeline Architecture & Method Chaining in Pandas?
Writing clean, maintainable data engineering code involves chaining transformation steps. Using **`df.pipe()`** and **`df.assign()`**, you can build modular, readable data processing pipelines that execute sequentially without cluttering memory with intermediate variables.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Method Chaining Architecture**: Covers `.pipe(custom_function)`.
- **Chunked File Iteration**: Covers `pd.read_csv(filepath, chunksize=10000)`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns


### 🔹 Method Chaining with `.pipe()`
Constructs a clean transformation pipeline on raw transactions. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `df.pipe(clean_dates).pipe(filter_fraud).pipe(calculate_metrics)`


In [2]:
def filter_valid_transactions(data):
    return data.dropna(subset=['transaction_amount'])

def add_fraud_flag_str(data):
    return data.assign(fraud_label=np.where(data['is_fraud'] == 1, 'FRAUD', 'LEGIT'))

clean_pipeline = df.pipe(filter_valid_transactions).pipe(add_fraud_flag_str)
print('Pipeline Processed Transactions Head:\n', clean_pipeline[['transaction_id', 'transaction_amount', 'fraud_label']].head(3))

Pipeline Processed Transactions Head:
   transaction_id  transaction_amount fraud_label
0       TX109326              607.78       LEGIT
1       TX106376             1819.11       FRAUD
2       TX103301               64.08       LEGIT


### 🔹 Chunked Processing with `pd.read_csv(chunksize=...)`
Streams `data/raw_transactions.csv` in 2,500-row chunks to calculate global metrics without memory spikes. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `for chunk in pd.read_csv('data/raw_transactions.csv', chunksize=2500): ...`


In [3]:
total_spend = 0.0
total_tx_count = 0
for chunk in pd.read_csv(csv_path, chunksize=2500):
    total_spend += chunk['transaction_amount'].sum()
    total_tx_count += len(chunk)
print(f'Processed {total_tx_count} transactions across chunks. Total Revenue: ${total_spend:,.2f}')

Processed 15000 transactions across chunks. Total Revenue: $14,326,935.50


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: PyArrow Engine Zero-Copy Performance

**Approach:** Ingest `raw_transactions.csv` using the PyArrow engine (`dtype_backend='pyarrow'`) for multi-threaded parsing.
**Syntax:** `pd.read_csv('data/raw_transactions.csv', engine='pyarrow', dtype_backend='pyarrow')`


In [4]:
t0 = time.perf_counter()
df_arrow = pd.read_csv(csv_path, engine='pyarrow', dtype_backend='pyarrow')
t_arrow = time.perf_counter() - t0
print(f'PyArrow Ingestion Time: {t_arrow*1000:.2f} ms ({df_arrow.shape[0]} rows)')

PyArrow Ingestion Time: 36.58 ms (15000 rows)


### 🔍 Scenario: Multi-File End-to-End Fintech Data Pipeline with `.pipe()`
Build a modular production data pipeline that ingests transactions, merges customer KYC profiles and merchant metadata, filters valid records, computes fees, and flags high-risk transactions.


In [5]:
# Modular Multi-File Fintech Pipeline
data_dir = 'data' if os.path.exists('data/customers.csv') else '../data'
df_cust = pd.read_csv(f'{data_dir}/customers.csv')
df_merch = pd.read_csv(f'{data_dir}/merchants.csv')

def stage_clean_transactions(data):
    return data.dropna(subset=['transaction_amount']).copy()

def stage_enrich_customers(data, customers):
    return data.merge(
        customers[['customer_id', 'kyc_status', 'risk_tier', 'credit_score', 'account_tier']],
        on='customer_id',
        how='left'
    )

def stage_enrich_merchants(data, merchants):
    return data.merge(
        merchants[['merchant_id', 'merchant_name', 'category', 'interchange_fee_pct']],
        on='merchant_id',
        how='left'
    )

def stage_calculate_revenue_and_flags(data):
    return (
        data.assign(
            fee_revenue=lambda d: d['transaction_amount'] * d['interchange_fee_pct'],
            is_high_exposure=lambda d: (d['transaction_amount'] > 1000) & (d['risk_tier'].isin(['High', 'Critical']))
        )
    )

# Execute pipeline using method chaining
processed_fintech_df = (
    df
    .pipe(stage_clean_transactions)
    .pipe(stage_enrich_customers, customers=df_cust)
    .pipe(stage_enrich_merchants, merchants=df_merch)
    .pipe(stage_calculate_revenue_and_flags)
)

print(f"Pipeline Complete: {processed_fintech_df.shape[0]} rows, {processed_fintech_df.shape[1]} columns")
print(processed_fintech_df[['transaction_id', 'first_name_dummy' if 'first_name_dummy' in processed_fintech_df else 'customer_id', 'kyc_status', 'category', 'fee_revenue', 'is_high_exposure']].head())


Pipeline Complete: 14251 rows, 20 columns
  transaction_id customer_id    kyc_status                   category  \
0       TX109326      C55082       Pending    Electronics & Computers   
1       TX106376      C76616      Verified    Crypto & Digital Assets   
2       TX103301      C65296      Verified              Food & Dining   
3       TX110701      C42098  Under Review          Travel & Airlines   
4       TX103284      C97782      Verified  Financial Services & SaaS   

   fee_revenue  is_high_exposure  
0    13.006492             False  
1    74.583510             False  
2     1.409760             False  
3    29.438451             False  
4    17.618472             False  
